# Olist Seller Agent

# Exploratório de Dados: Recorrência de Compra e Perfil de Retenção de Clientes
> **Projeto:** Transformação Digital Olist via IA Agêntica  
> **Foco do Notebook:** Análise de histórico de compras por cliente (`olist_customers_dataset`), intervalo entre pedidos e ticket médio.

---
### Objetivo deste Notebook
Este notebook analisa os padrões comportamentais dos clientes satisfeitos para comprovar que o baixo índice de recompra decorre da ausência de engajamento ativo pós-venda, justificando a criação do **Agente de Reativação de Clientes**.

Neste notebook, realizamos:
1. **Mensuração da Taxa de Recompra:** Cálculo da proporção de clientes recorrentes (*sessão única*, *até 7 dias* e *após 8 dias*).
2. **Isolamento de Variáveis de Insatisfação:** Cruzamento entre nota dada na primeira avaliação e taxa de retorno (provando que clientes satisfeitos também não retornam sem estímulo).
3. **Análise de Concentração de Categorias e Basket Size:** Identificação da ausência de *cross-selling* no momento do checkout e mapeamento de janelas ideais para reengajamento.

## Conectar o Google Drive

Para rodar no Colab, precisamos montar o Drive para acessar os CSVs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Carregar dados relevantes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PASTA = Path('/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/')

# Paleta única do relatório
AZUL, LARANJA, VERMELHO = '#2a78d6', '#eb6834', '#e34948'
CINZA, HAIR, TINTA = '#898781', '#e1e0d9', '#101317'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 13, 'axes.titleweight': '600', 'axes.titlelocation': 'left',
    'axes.grid': True, 'axes.axisbelow': True,
    'grid.color': HAIR, 'grid.alpha': 0.6,
    'font.size': 10, 'figure.dpi': 110,
})

orders = pd.read_csv(PASTA / 'olist_orders_dataset.csv', parse_dates=[
    'order_purchase_timestamp', 'order_delivered_customer_date',
    'order_estimated_delivery_date'])
customers = pd.read_csv(PASTA / 'olist_customers_dataset.csv')
order_items = pd.read_csv(PASTA / 'olist_order_items_dataset.csv')
order_reviews = pd.read_csv(PASTA / 'olist_order_reviews_dataset.csv')

def brl(v, pos=None):
    if abs(v) >= 1e6: return f'R$ {v/1e6:.1f} mi'.replace('.', ',')
    if abs(v) >= 1e3: return f'R$ {v/1e3:.0f} mil'
    return f'R$ {v:.0f}'

print(f'✅ {len(orders):,} pedidos carregados'.replace(',', '.'))

# Olist Intelligent Marketplace
## Onde a IA cria valor — e onde ela não é a resposta

**Tech Challenge · Fase 1 — Fundamentos de IA e Agentic AI**
Base analisada: 99.441 pedidos · jan/2017 a ago/2018 · Brazilian E-Commerce Public Dataset by Olist

---

### Sumário executivo

A Olist parou de crescer porque cresceu.

Entre 2017 e o primeiro semestre de 2018 a receita mais que dobrou. Depois, oito meses
de platô. Testamos 18 hipóteses para explicar o que travou, e três coisas ficaram claras:

**1. A aquisição chegou ao teto e não existe segunda venda.** Apenas 3,4% dos clientes
voltam. Não é insatisfação — quem deu nota 1 recompra tanto quanto quem deu nota 5.
É ausência de mecanismo: ninguém convida o cliente a voltar, e 90% dos pedidos têm um
único item.

**2. A operação quebra exatamente nos picos que a receita produz.** A correlação entre
receita mensal e atraso é +0,52. Os melhores meses de venda são os piores meses de entrega.

**3. Metade do problema é invisível aos indicadores.** 69% das reclamações não tiveram
atraso nenhum — são defeito, produto errado, entrega incompleta. Sinais que só existem
em texto livre e que ninguém lê em escala.

**A recomendação:** quatro problemas, quatro donos. Dois se resolvem com configuração e
modelo estatístico — sem IA. Dois exigem agentes, porque a entrada é texto e o espaço
de ação é aberto. Propor IA onde uma planilha resolve custa credibilidade; por isso
chegamos a **três agentes bem justificados**, não a seis inflados.

| Problema | Dono | Solução | Tecnologia |
|---|---|---|---|
| Pesquisa disparada antes da entrega | Operações | Mudar gatilho para "entrega confirmada" | ⚙️ Configuração |
| Promessa uniforme onde o risco não é | Operações | Prazo por rota calibrado no P90 | 📐 Modelo estatístico |
| Cliente não volta | Comercial | Régua de relacionamento + recomendação | ⚙️ + 🧠 GenAI |
| Capacidade reativa nos picos | Logística | Previsão de carga com detecção de anomalia | 📐 Série temporal |
| 100 vendedores concentram o atraso | Logística | Diagnóstico e negociação de plano | 🤖 **Agente** |
| 29% das reclamações invisíveis | Curadoria | Classificação e roteamento de texto | 🤖 **Agente** |
| "Cadê meu pedido?" antes da reclamação | Atendimento | Aviso proativo + diálogo | 🤖 **Agente** |

---
# PARTE I — O DIAGNÓSTICO

## 1. O fato: a receita parou

Crescimento de +178% no 1º semestre de 2018 contra 2017. Depois, platô entre
R\$ 840 mil e R\$ 990 mil por mês, por oito meses seguidos.

O ticket médio ficou estável em R\$ 130–150 o tempo todo. Isso significa que **todo o
crescimento veio de volume, nunca de valor por cliente** — e é a primeira pista de para
onde olhar.

In [ ]:
# ══════════ G1 — Receita mensal com o platô destacado ══════════
itens = order_items.merge(orders[['order_id', 'order_status',
                                  'order_purchase_timestamp']], on='order_id')
itens = itens[~itens['order_status'].isin(['canceled', 'unavailable'])]
itens['mes'] = itens['order_purchase_timestamp'].dt.to_period('M').dt.to_timestamp()

receita = itens.groupby('mes')['price'].sum()['2017-01':'2018-08']

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(receita.index, receita.values, color=AZUL, linewidth=2, zorder=3)
ax.fill_between(receita.index, receita.values, color=AZUL, alpha=0.10, zorder=2)
ax.axvspan(pd.Timestamp('2018-01-01'), receita.index[-1], color=HAIR, alpha=0.5, zorder=0)

ax.annotate('platô: 8 meses sem crescer', xy=(pd.Timestamp('2018-04-15'), receita.max()*0.42),
            ha='center', color=CINZA, fontsize=10)
pico = receita.idxmax()
ax.annotate(f'Black Friday\n{brl(receita.max())}', xy=(pico, receita.max()),
            xytext=(0, 10), textcoords='offset points', ha='center',
            fontweight='700', linespacing=1.4)

ax.set_title('A receita triplicou em 2017 — e parou em 2018')
ax.set_ylabel('Receita (produtos, sem frete)')
ax.set_ylim(0, receita.max()*1.28)
ax.yaxis.set_major_formatter(plt.FuncFormatter(brl))
ax.grid(axis='x', visible=False)
plt.tight_layout(); plt.show()

2. Duas explicações concorrentes
Se o crescimento vinha só de volume, o platô tem duas causas possíveis — e só duas:

Rota A — o motor de aquisição chegou ao limite e não há segunda venda. Cada real de receita depende de um cliente novo. Quando a captação satura, a receita satura.

Rota B — a operação não absorve o volume que a receita produz. Se entregar mais custa qualidade, existe um teto operacional que nenhum investimento em marketing atravessa.

As duas rotas levam a soluções opostas. A Rota A é um problema comercial: falta régua de relacionamento. A Rota B é um problema logístico: falta planejamento de capacidade. Investir na errada é caro.

As seções 3 e 4 testam cada uma. Antecipando o resultado: as duas se confirmam, por mecanismos independentes. É por isso que a resposta da Olist não pode ser um único projeto.

3. Rota A — o cliente não volta, e não é por insatisfação
96,9% dos clientes compraram uma única vez. Esse grupo responde por 94% da receita.

Antes de chamar isso de problema, era preciso descartar a explicação mais confortável: a base é jovem, faltou tempo. Ela é apenas parcialmente verdadeira.

In [ ]:
# ══════════ Base de recompra (usada nos G3 a G6) ══════════
base = orders.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id')
base = base[base['order_status'] != 'canceled'].sort_values('order_purchase_timestamp')
base['ordem'] = base.groupby('customer_unique_id').cumcount() + 1
FIM = base['order_purchase_timestamp'].max()

primeira = base[base['ordem'] == 1].copy()
n_ped = base.groupby('customer_unique_id')['order_id'].nunique()
primeira['recomprou'] = primeira['customer_unique_id'].map(n_ped) > 1
primeira['meses'] = (FIM - primeira['order_purchase_timestamp']).dt.days / 30.44
primeira['atraso'] = (primeira['order_delivered_customer_date']
                      - primeira['order_estimated_delivery_date']).dt.days

p2 = base[base['ordem'] == 2][['customer_unique_id', 'order_purchase_timestamp']]
ciclo = primeira[['customer_unique_id', 'order_purchase_timestamp']].merge(
    p2, on='customer_unique_id', suffixes=('_1', '_2'))
ciclo['dias'] = (ciclo['order_purchase_timestamp_2']
                 - ciclo['order_purchase_timestamp_1']).dt.days

# ══════════ G3 — Curva acumulada do tempo até a 2ª compra ══════════
lim = [0, 7, 15, 30, 60, 90, 120, 180, 270, 365, 540]
acum = [(ciclo['dias'] <= d).mean()*100 for d in lim]

fig, ax = plt.subplots(figsize=(9.5, 4.8))
ax.plot(lim, acum, color=AZUL, linewidth=2, marker='o', markersize=5, zorder=3)
ax.axhline(97, color=HAIR, linestyle='--', linewidth=1.2)
ax.axvline(365, color=HAIR, linestyle='--', linewidth=1.2)
ax.annotate('97% das recompras já\naconteceram em 12 meses', xy=(365, 97),
            xytext=(-15, -50), textcoords='offset points', ha='right',
            fontweight='700', linespacing=1.4,
            arrowprops=dict(arrowstyle='->', color=CINZA))
ax.annotate(f'mediana: {ciclo["dias"].median():.0f} dias', xy=(28, 51),
            xytext=(18, -8), textcoords='offset points', color='#4a535f')
ax.set_title('Quem volta, volta rápido — e o ciclo já se esgotou')
ax.set_xlabel('Dias desde a primeira compra'); ax.set_ylabel('% das recompras já ocorridas')
ax.set_xlim(0, 540); ax.set_ylim(0, 105)
plt.tight_layout(); plt.show()

# ══════════ G4 — Decomposição da recompra ══════════
mad = primeira[primeira['meses'] >= 12]
cm = ciclo[ciclo['customer_unique_id'].isin(mad['customer_unique_id'])]
n = len(mad)
vals = [(cm['dias'] <= 1).sum()/n*100,
        ((cm['dias'] > 1) & (cm['dias'] <= 7)).sum()/n*100,
        (cm['dias'] > 7).sum()/n*100]

fig, ax = plt.subplots(figsize=(7.5, 4.6))
b = ax.bar(['Mesma sessão\n(até 1 dia)', 'Até 7 dias', 'Retenção real\n(8+ dias)'],
           vals, color=[CINZA, LARANJA, AZUL], width=0.6)
for bb, v in zip(b, vals):
    ax.annotate(f'{v:.2f}%', xy=(bb.get_x()+bb.get_width()/2, v), xytext=(0, 5),
                textcoords='offset points', ha='center', fontsize=11, fontweight='700')
ax.set_title(f'A recompra de {sum(vals):.2f}% cai para {vals[2]:.2f}% sem a mesma sessão')
ax.set_ylabel('% dos clientes (12+ meses de janela)')
ax.set_ylim(0, max(vals)*1.25); ax.grid(axis='x', visible=False)
plt.tight_layout(); plt.show()

A janela importa (a recompra vai de 1,55% para 5,10% quando se olha só quem teve 12 meses), mas 97% das recompras acontecem dentro de 12 meses. O ciclo já se esgotou. O teto real é 5,1%, não algo que o tempo vá corrigir.

E há um ajuste que muda o número: a mediana entre 1ª e 2ª compra é de 28 dias, mas 27,6% das segundas compras acontecem no mesmo dia — carrinho dividido, não retorno. Excluindo o que ocorre em até 7 dias, a retenção genuína é 3,43%.

Sobra a pergunta: por que não voltam? Testamos as três explicações óbvias.

In [ ]:
# ══════════ G5 — Recompra por entrega, COM barras de erro ══════════
m12 = primeira[(primeira['meses'] >= 12)
               & primeira['order_delivered_customer_date'].notna()].copy()
m12['exp'] = pd.cut(m12['atraso'], [-999, 0, 7, 999],
                    labels=['Chegou\nno prazo', 'Atraso\n1 a 7 d', 'Atraso\n8+ d'])

g = m12.groupby('exp', observed=True)['recomprou'].agg(['size', 'mean'])
g['pct'] = g['mean']*100
g['ic'] = 1.96*np.sqrt(g['mean']*(1-g['mean'])/g['size'])*100

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.bar(g.index.astype(str), g['pct'], yerr=g['ic'], capsize=7,
       color=[AZUL, LARANJA, VERMELHO], width=0.6,
       error_kw=dict(ecolor='#4a535f', lw=1.5))
for i, (p, n_, ic) in enumerate(zip(g['pct'], g['size'], g['ic'])):
    ax.annotate(f'{p:.2f}%\nn={n_:,}'.replace(',', '.'), xy=(i, p+ic),
                xytext=(0, 6), textcoords='offset points', ha='center',
                fontsize=9.5, fontweight='700', linespacing=1.4)
ax.set_title('Os intervalos se sobrepõem — a hipótese não pôde ser aceita')
ax.set_ylabel('% que recomprou'); ax.set_ylim(0, 8)
ax.grid(axis='x', visible=False)
plt.tight_layout(); plt.show()

# ══════════ G6 — Recompra por nota da 1ª compra ══════════
pr = primeira[primeira['meses'] >= 12].merge(
    order_reviews.drop_duplicates('order_id')[['order_id', 'review_score']],
    on='order_id', how='left')
gn = pr.dropna(subset=['review_score']).groupby('review_score')['recomprou'].mean()*100

fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.bar(gn.index.astype(int).astype(str), gn.values, color=AZUL, width=0.6)
for i, v in enumerate(gn.values):
    ax.annotate(f'{v:.2f}%', xy=(i, v), xytext=(0, 5), textcoords='offset points',
                ha='center', fontweight='700')
ax.set_title('A nota não prevê recompra — a linha reta é o achado')
ax.set_xlabel('Nota dada na primeira compra'); ax.set_ylabel('% que recomprou')
ax.set_ylim(0, 7); ax.grid(axis='x', visible=False)
plt.tight_layout(); plt.show()

# ══════════ G7 — Itens por pedido ══════════
ipp = order_items.groupby('order_id').size().value_counts(normalize=True).sort_index()*100
ipp = pd.concat([ipp[:3], pd.Series({'4+': ipp[4:].sum()})])

fig, ax = plt.subplots(figsize=(7, 4.4))
ax.bar(ipp.index.astype(str), ipp.values,
       color=[VERMELHO] + [AZUL]*(len(ipp)-1), width=0.6)
for i, v in enumerate(ipp.values):
    ax.annotate(f'{v:.1f}%', xy=(i, v), xytext=(0, 5), textcoords='offset points',
                ha='center', fontweight='700')
ax.set_title('90% dos pedidos têm um único item — não existe cross-sell')
ax.set_xlabel('Itens no pedido'); ax.set_ylabel('% dos pedidos')
ax.set_ylim(0, 105); ax.grid(axis='x', visible=False)
plt.tight_layout(); plt.show()

O problema da falta de recompra e da baixa retenção de clientes não está relacionado a questões logísticas ou regionais, uma vez que o comportamento é uniforme em todos os estados, descartando a hipótese de restrições de frete ou concentração geográfica.

In [ ]:
# ══════════ Gx — Recompra por Região (Retenção Real - 8+ dias) ══════════

# Adicionar a coluna 'regiao' ao DataFrame
# Certificar-se de que 'REGIAO' está definido, como em G10 (cell N7RNCq83FMK3)
REGIAO = {**dict.fromkeys(['AC','AP','AM','PA','RO','RR','TO'], 'Norte'),
          **dict.fromkeys(['AL','BA','CE','MA','PB','PE','PI','RN','SE'], 'Nordeste'),
          **dict.fromkeys(['DF','GO','MT','MS'], 'Centro-Oeste'),
          **dict.fromkeys(['ES','MG','RJ','SP'], 'Sudeste'),
          **dict.fromkeys(['PR','RS','SC'], 'Sul')}

# CORREÇÃO: Mergear 'primeira' com 'customers' usando 'customer_id' para evitar duplicação
# e garantir que cada primeira compra seja associada ao seu estado correto.
primeira_com_estado_corrigido = primeira.merge(customers[['customer_id', 'customer_state']],
                                     on='customer_id', how='left')

# Filtrar para clientes com janela de tempo de pelo menos 12 meses (similar a G5)
mad_com_estado = primeira_com_estado_corrigido[primeira_com_estado_corrigido['meses'] >= 12].copy()

mad_com_estado['regiao'] = mad_com_estado['customer_state'].map(REGIAO)

# Identificar customer_unique_id que tiveram recompra real (acima de 7 dias)
# O DataFrame `ciclo` contém a informação de dias entre a primeira e a segunda compra.
real_recompra_customer_ids = ciclo[ciclo['dias'] > 7]['customer_unique_id'].unique()

# Criar uma nova coluna em mad_com_estado para indicar a recompra real (8+ dias)
mad_com_estado['recomprou_real'] = mad_com_estado['customer_unique_id'].isin(real_recompra_customer_ids)

# Calcular a porcentagem de recompra real por região
recompra_por_regiao = mad_com_estado.groupby('regiao')['recomprou_real'].mean() * 100
recompra_por_regiao = recompra_por_regiao.sort_values(ascending=False).to_frame('pct_recompra_real')

fig, ax = plt.subplots(figsize=(10, 6))

# Cores para o gráfico, destacando as regiões principais se necessário, mas aqui usaremos uma única cor
ax.barh(recompra_por_regiao.index, recompra_por_regiao['pct_recompra_real'],
        color=AZUL, height=0.7)

# Adicionar os valores percentuais nas barras
for i, (regiao, pct) in enumerate(recompra_por_regiao.iterrows()):
    ax.annotate(f'{pct['pct_recompra_real']:.2f}%', xy=(pct['pct_recompra_real'], i),
                xytext=(5, 0), textcoords='offset points', va='center',
                fontsize=9.5, fontweight='700', color=TINTA)

ax.set_title('Recompra Real de clientes por Região (8+ dias, após 12+ meses de janela)')
ax.set_xlabel('% de clientes que recompraram (retenção real)')
ax.set_ylabel('Região')
ax.set_xlim(0, recompra_por_regiao['pct_recompra_real'].max() * 1.2)
ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.show()

display(recompra_por_regiao.round(2))